# Analyse des vœux impossibles
Ce notebook charge les fichiers d'entrée (`data/samples`) et liste les vœux pour lesquels aucun des choix n'est accessible selon `rules.Feasibility`. Il produit `out/unassigned_reasons.csv` avec les motifs détaillés.

In [1]:
from pathlib import Path
import sys
import pandas as pd
# Ajuster le chemin vers la racine du dépôt
REPO = Path("d:/school/course-allocation")
sys.path.insert(0, str(REPO))
# Imports du projet
from src.data.loaders import build_campaign
from src.rules.feasibility import Feasibility

students_path = REPO / 'data' / 'samples' / 'etudiants_anonymises.csv'
campaign_path = REPO / 'data' / 'samples' / 'campagne_synthetique.csv'
ecue_path = REPO / 'src' / 'data' / 'default_ecue.csv'

campaign = build_campaign(students_path, campaign_path, ecue_path)
print(f'Loaded campaign {campaign.id_campagne}, {len(campaign.students)} students, {len(campaign.occurrences)} occurrences, {len(campaign.voeux)} voeux')

Loaded campaign 544, 320 students, 63 occurrences, 2731 voeux


In [2]:
# Analyse : pour chaque vœu, vérifier si au moins une occurrence est accessible ; sinon collecter les raisons
f = Feasibility()
rows = []
for v in campaign.voeux:
    s = campaign.students.get(v.id_student)
    if s is None:
        rows.append({
            'id_student': v.id_student, 'id_demande': v.id_demande,
            'ranked_occurrences': ';'.join(v.ranked_occurrences),
            'reasons': 'élève absent des fichiers students', 'per_occurrence': ''
        })
        continue
    accessible_any = False
    occ_reasons = {}
    for occ_id in v.ranked_occurrences:
        o = campaign.occurrences.get(occ_id)
        if o is None:
            occ_reasons[occ_id] = ['occurrence inconnue']
            continue
        msgs = f.check(s, o)
        if not msgs:
            accessible_any = True
            break
        occ_reasons[occ_id] = msgs
    if not accessible_any:
        flat = sorted({r for reasons in occ_reasons.values() for r in reasons})
        rows.append({
            'id_student': v.id_student, 'id_demande': v.id_demande,
            'ranked_occurrences': ';'.join(v.ranked_occurrences),
            'reasons': ';'.join(flat), 'per_occurrence': str(occ_reasons)
        })

df = pd.DataFrame(rows)
display(df.head(100))
out_path = REPO / 'out' / 'unassigned_reasons.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False, sep=';')
print('Saved to', out_path)

,id_student,id_demande,ranked_occurrences,reasons,per_occurrence
0,300002,54409,36790;36792,créneau Lu-am occupé par filière CYBER;jour Ve...,{'36790': ['créneau Lu-am occupé par filière C...
1,300003,54403,35307,créneau Lu-am occupé par filière SE,{'35307': ['créneau Lu-am occupé par filière S...
2,300003,54405,36858;35412,créneau Lu-am occupé par filière SE;jour Mercr...,{'36858': ['créneau Lu-am occupé par filière S...
3,300004,54401,35308,jour Mercredi en entreprise,{'35308': ['jour Mercredi en entreprise']}
4,300004,54402,36813,jour Mercredi en entreprise,{'36813': ['jour Mercredi en entreprise']}
...,...,...,...,...,...
95,300150,54402,36815,créneau Ve-am occupé par filière SLR,{'36815': ['créneau Ve-am occupé par filière S...
96,300154,54407,35597,créneau Lu-am occupé par filière SLR,{'35597': ['créneau Lu-am occupé par filière S...
97,300155,54407,36789,occurrence FISEA réservée aux apprentis,{'36789': ['occurrence FISEA réservée aux appr...
98,300155,54408,36775,occurrence FISEA réservée aux apprentis,{'36775': ['occurrence FISEA réservée aux appr...


Saved to d:\school\course-allocation\out\unassigned_reasons.csv


**Prochaines étapes** :
- Exécuter ce notebook pour générer `out/unassigned_reasons.csv`.
- Si tu veux, j'exécute les cellules et te montre les premières lignes ici.